# Pulmonary CFD: Runpod or a Linux VM

This notebook runs locally with a **Python 3 kernel** and controls a Linux x86_64 Ubuntu 20.04/22.04 host over direct SSH. It uploads `0084_0001.zip`, builds pinned SimVascular/MPI tools, benchmarks 401 steps, and can resume the full 8,125-step run. A Google Cloud VM or university workstation uses the same SSH path.

The original mesh, inflow and outlet conditions are preserved. Benchmark timing and solver execution do not establish scientific convergence.

**Before running:** choose a funded Runpod pod or a VM, configure the connection below, and keep the companion Python/shell files beside this notebook. SSH/SCP must be installed locally. SSH passwords are not supported; use a private key or your SSH agent. API keys are entered privately and are never stored.

New Runpod provisioning and the full solve are disabled by default. A free Google Cloud trial is quota-limited; see `README.md` before choosing a VM. Free Colab's GPU does not accelerate this CPU/MPI solver.


In [ ]:
from pathlib import Path
import getpass
import json
import os
import shlex
import sys
import time

# Find companion scripts whether Jupyter started from the repo or this directory.
SCRIPTS = next((p for p in [Path.cwd(), Path.cwd() / "scripts/cfd"]
                if (p / "runpod_cfd.py").is_file()), None)
if SCRIPTS is None:
    raise FileNotFoundError("Open this notebook from the repo root or scripts/cfd")
SCRIPTS = SCRIPTS.resolve()
sys.path.insert(0, str(SCRIPTS))
from cfd_case import atomic_json
from runpod_cfd import Remote, RunpodAPI

ARCHIVE = Path.home() / "Downloads/0084_0001.zip"
LOCAL_OUTPUT = SCRIPTS.parents[1] / "outputs/cfd_runpod"
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

PROVIDER = "ssh"       # "ssh" for any VM; "runpod" for Runpod API discovery/provisioning
SSH_HOST = ""          # Enter the VM public IP or Runpod's DIRECT SSH host
SSH_PORT = 22          # Runpod's direct SSH port is usually not 22; use its Connect tab
SSH_USER = "root"      # For Google Cloud use your VM's Linux username
SSH_KEY = Path.home() / ".ssh/id_ed25519"
REMOTE_ROOT = "/workspace/dynamag"  # For a VM: /home/YOUR_LINUX_USERNAME/dynamag
POD_ID = ""            # Reuse a saved pod if PROVIDER="runpod"
CREATE_RUNPOD = False  # Set True only when you intend to rent a new CPU pod
VCPUS = 32
NETWORK_VOLUME_ID = None  # Optional existing Runpod network volume for persistent storage
POD_LIFETIME_SECONDS = 6 * 3600  # Container lifetime guard for a benchmark pod
HOURLY_COMPUTE_USD = 0.96       # Replace with the selected VM's current resource price
# Trial credits cover resource cost; they do not change the VM's underlying hourly rate.

JOB_TAG = "v1"         # Change only to retry a finished/failed job after fixing its cause
RUN_BENCHMARK = True
RUN_FULL_SOLVE = False
POSTPROCESS_FULL = False
BACKUP_CHECKPOINTS = True
RESTORE_BACKUP = None  # Path to downloaded checkpoints.tar.gz; use an empty REMOTE_ROOT
TERMINATE_CREATED_POD = False  # Requires a verified local checkpoint backup

if not ARCHIVE.is_file():
    raise FileNotFoundError(ARCHIVE)
if not 0 < VCPUS <= 32 or not isinstance(VCPUS, int):
    raise ValueError("Use an integer rank count between 1 and 32")

if PROVIDER == "runpod" and CREATE_RUNPOD and RUN_FULL_SOLVE and POD_LIFETIME_SECONDS < 25 * 3600:
    raise ValueError("For setup plus a full solve, increase POD_LIFETIME_SECONDS to at least 25 hours and use persistent storage")


## Connect or provision

For an existing VM, set `SSH_HOST`, `SSH_PORT`, `SSH_USER`, and `REMOTE_ROOT`. For Runpod API discovery, set `PROVIDER="runpod"` and `POD_ID`, or explicitly enable creation. The default CPU pod costs roughly $0.96/hour plus disk; creation reads a live CPU quote first. Authentication is scoped to the new pod, without changing account-wide SSH keys.

The container lifetime guard ends the container, **not** the pod's storage billing. Download your backup and terminate the pod after benchmarking. Use a longer lifetime and persistent storage for a full solve. Never stop/delete an ephemeral-disk pod before downloading checkpoints.


In [ ]:
api = None
state_path = LOCAL_OUTPUT / "connection.json"
created_here = False
saved = json.loads(state_path.read_text()) if state_path.exists() else {}
if PROVIDER == "runpod":
    api = RunpodAPI(os.environ.get("RUNPOD_API_KEY") or getpass.getpass("Runpod API key: "))
    quote = api.cpu_quote(VCPUS)
    print("Live CPU quote:", quote, "plus disk/network-volume charges")
    HOURLY_COMPUTE_USD = quote["compute_usd_per_hour"]
    POD_ID = POD_ID or saved.get("pod_id", "")
    if not POD_ID:
        if not CREATE_RUNPOD:
            raise ValueError("Set an existing POD_ID or explicitly enable CREATE_RUNPOD")
        public_key = SSH_KEY.with_suffix(SSH_KEY.suffix + ".pub").read_text().strip()
        pod = api.create_cpu(public_key, VCPUS, POD_LIFETIME_SECONDS, NETWORK_VOLUME_ID)
        POD_ID = pod["id"]
        created_here = True
        # Record the identity immediately so interruptions do not create duplicate paid pods.
        atomic_json(state_path, {"pod_id": POD_ID, "created_by_notebook": True})
    deadline = time.monotonic() + 600
    while True:
        pod = api.pod(POD_ID)
        if (pod.get("ssh") or {}).get("direct"):
            remote = Remote.from_pod(pod, SSH_KEY, LOCAL_OUTPUT / "known_hosts")
            break
        if time.monotonic() >= deadline:
            raise TimeoutError("Pod did not expose direct SSH; inspect its console, then reconnect using the saved ID")
        time.sleep(10)
    print("Connected pod:", POD_ID)
else:
    if not SSH_HOST:
        raise ValueError("Set SSH_HOST to your Linux VM or Runpod direct SSH endpoint")
    remote = Remote(SSH_HOST, SSH_PORT, SSH_USER, SSH_KEY, LOCAL_OUTPUT / "known_hosts")

# The endpoint may appear before sshd finishes installing. Bound connection attempts.
deadline = time.monotonic() + 600
while True:
    try:
        hardware = json.loads(remote.run("python3 -c " + shlex.quote('import json,os,platform; print(json.dumps({"cpus": len(os.sched_getaffinity(0)), "architecture": platform.machine(), "system": platform.system()}))')))
        break
    except RuntimeError:
        if time.monotonic() >= deadline:
            raise
        time.sleep(10)
if hardware["architecture"] != "x86_64" or hardware["system"] != "Linux":
    raise ValueError("The setup requires Linux x86_64")
if VCPUS > hardware["cpus"]:
    raise ValueError(f"Only {hardware['cpus']} CPUs are available; reduce VCPUS")
print("Remote hardware:", hardware)
atomic_json(state_path, {"pod_id": POD_ID or None,
    "created_by_notebook": created_here or (PROVIDER == "runpod" and saved.get("pod_id") == POD_ID and saved.get("created_by_notebook", False)),
    "host": remote.host, "port": remote.port, "user": remote.user,
    "remote_root": REMOTE_ROOT, "hourly_compute_usd": HOURLY_COMPUTE_USD})


## Upload, optionally restore, and install

The ZIP is verified with SHA-256 after transfer. Extraction refuses to overwrite an existing case. Restoring requires an empty workspace and rebuilds the toolchain; restore does not change the checkpoint's MPI rank count.

Setup uses GCC 9, VTK 8.2.0, OpenMPI, and svSolver commit `47de08e103078587264ad56089a387435a46d3c0`. It runs detached and writes a log; notebook disconnection does not cancel it. Building the toolchain can take substantial time and is billed on a rented VM.


In [ ]:
if RESTORE_BACKUP is not None:
    print(remote.restore(Path(RESTORE_BACKUP), REMOTE_ROOT))
digest = remote.upload_workflow(SCRIPTS, ARCHIVE, REMOTE_ROOT)
print("Verified uploaded archive:", digest)
q = shlex.quote
runner = q(REMOTE_ROOT + "/scripts/cfd_case.py")
remote_archive = q(REMOTE_ROOT + "/inputs/" + ARCHIVE.name)
runs = q(REMOTE_ROOT + "/runs")
print(remote.run(f"python3 {runner} prepare {remote_archive} {q(REMOTE_ROOT + '/runs/production')} --sha256 {digest}"))
setup_name = "setup-" + JOB_TAG
# sudo -n fails clearly if this VM user lacks passwordless administrative access.
setup_command = f"sudo -n bash {q(REMOTE_ROOT + '/scripts/setup_svsolver.sh')} {q(REMOTE_ROOT + '/toolchain')}"
if SSH_USER == "root" or (PROVIDER == "runpod" and remote.user == "root"):
    setup_command = f"bash {q(REMOTE_ROOT + '/scripts/setup_svsolver.sh')} {q(REMOTE_ROOT + '/toolchain')}"
print(remote.launch(REMOTE_ROOT, setup_name, setup_command, timeout_seconds=7200))
remote.wait(REMOTE_ROOT, setup_name, seconds=7500)
remote.download(REMOTE_ROOT + "/toolchain/setup.json", LOCAL_OUTPUT / "setup.json")
source_env = f"source {q(REMOTE_ROOT + '/toolchain/env.sh')}; "


## Benchmark with a real resume and VTU export

This uses a separate copy of the case: 50 warmup steps, then resumes through step 401 (351 measured steps). The measured phase includes MPI startup and checkpoint I/O. Exporting the final VTU checks the postprocessing path. Re-executing a completed benchmark reads its saved report. An interrupted measured interval requires a fresh benchmark root for trustworthy timing; it is not silently treated as a complete sample.

The estimate covers only early-cycle flow and is not a guarantee for the entire cardiac waveform. Use a full-cycle benchmark if the estimate is close to a deadline.


In [ ]:
if RUN_BENCHMARK:
    benchmark_root = REMOTE_ROOT + "/runs/" + JOB_TAG
    benchmark_name = f"benchmark-{VCPUS}-" + JOB_TAG
    command = (source_env + f"python3 {runner} benchmark {remote_archive} {q(benchmark_root)} "
               + f"--ranks {VCPUS} --hourly {HOURLY_COMPUTE_USD} --warmup 50 --until 401")
    print(remote.launch(REMOTE_ROOT, benchmark_name, command, timeout_seconds=7200))
    remote.wait(REMOTE_ROOT, benchmark_name, seconds=7500)
    report_local = LOCAL_OUTPUT / f"benchmark_{VCPUS}_{JOB_TAG}.json"
    remote.download(benchmark_root + f"/benchmark_{VCPUS}/benchmark.json", report_local)
    report = json.loads(report_local.read_text())
    print(json.dumps(report, indent=2))
    print(f"Projected solve: {report['projected_full_solve_hours']:.2f} hours; "
          f"resource compute cost: ${report['projected_full_compute_usd']:.2f}")
    print("Compare this estimate with the time remaining AFTER installation/upload; allow margin for later-cycle flow.")


## Full solve and second-cycle export (opt-in)

Set `RUN_FULL_SOLVE=True` after inspecting the benchmark. Production checkpoints are independent of the benchmark. A resume keeps the same MPI rank count, finds the newest complete checkpoint across all ranks, and runs only the remaining steps through step 8,125. Partial or truncated newer checkpoints are skipped. Concurrent runs of the same case are rejected.

`numstart.dat` inside `N-procs_case` is the active restart pointer. The runner handles final checkpoints whose pointer was not advanced by legacy svSolver. `svpost` reads that partition directory and verifies each expected VTU frame.


In [ ]:
production = REMOTE_ROOT + "/runs/production/0084_0001"
if RUN_FULL_SOLVE:
    full_name = "full-" + JOB_TAG
    command = source_env + f"python3 {runner} run {q(production)} --ranks {VCPUS} --until 8125"
    print(remote.launch(REMOTE_ROOT, full_name, command, timeout_seconds=23 * 3600))
    remote.wait(REMOTE_ROOT, full_name, seconds=23 * 3600 + 60)
    remote.download(production + "/last_run.json", LOCAL_OUTPUT / "full_run.json")
if POSTPROCESS_FULL:
    post_name = "postprocess-" + JOB_TAG
    command = source_env + f"python3 {runner} postprocess {q(production)} --ranks {VCPUS}"
    print(remote.launch(REMOTE_ROOT, post_name, command, timeout_seconds=3600))
    remote.wait(REMOTE_ROOT, post_name, seconds=3660)


## Download checkpoints and clean up

Keep the downloaded archive and `backup.json` before stopping or deleting the remote machine. The backup contains input files, rank partitions, logs, benchmark reports and VTUs. For a Google Cloud VM, stop/delete it in its console only after verifying this download. For a Runpod pod created here, the optional cleanup cell terminates it only after a successful backup. Network volumes are separate resources and remain billable until removed independently.

If a previous cell fails, execute the backup cell manually to preserve available checkpoints; inspect the job log, fix the cause, then use a new `JOB_TAG` for a new job. Resume the same production case with the same rank count. A notebook disconnection does not require restarting a still-running job.


In [ ]:
backup_path = None
if BACKUP_CHECKPOINTS:
    backup_path = remote.backup(REMOTE_ROOT, LOCAL_OUTPUT)
    print("Verified local checkpoint backup:", backup_path)


In [ ]:
if TERMINATE_CREATED_POD:
    state = json.loads(state_path.read_text())
    if not api or not POD_ID or not state.get("created_by_notebook") or state.get("pod_id") != POD_ID:
        raise ValueError("Automatic cleanup only applies to the specific pod created by this notebook")
    if backup_path is None or not backup_path.is_file():
        raise ValueError("Download and verify a checkpoint backup before termination")
    api.terminate(POD_ID)
    atomic_json(state_path, {"terminated_pod_id": POD_ID, "created_by_notebook": False})
    print("Terminated pod:", POD_ID)
